# Brisbane Water Quality — Time Series Analysis
## Итоговое задание по дисциплине «Анализ временных рядов»

**Датасет**: Brisbane Water Quality (30-минутные наблюдения, 2023-08-04 — 2023-10-06)  
**Цель**: Прогнозирование температуры воды (горизонт 48 шагов = 24 часа)  
**Студент**: Александр Филипов


## 0. Импорт библиотек

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

TARGET     = "Temperature"
FREQ       = "30min"
HORIZON    = 48        # 24 hours
SEASON_LEN = 48
N_WINDOWS  = 3
SEED       = 42
np.random.seed(SEED)
print("Libraries loaded OK")


## 1. Задача №1 — Подготовка данных и EDA
### 1.1 Загрузка и очистка данных

**Описание ВР**: Многомерный временной ряд качества воды реки Брисбен.  
Измерения каждые 30 минут: температура, pH, кислород, мутность, хлорофилл, соленость.  
**Постановка задачи**: Прогноз температуры воды на 24 часа вперёд (офлайн, глобальная модель).  
**Горизонт прогнозирования**: H=48 шагов (48×30мин = 24ч).  
**Метрики**: MAE, RMSE, MAPE%.


In [ ]:
df_raw = pd.read_csv("data/brisbane_water_quality.csv", parse_dates=["Timestamp"])
df_raw = df_raw.sort_values("Timestamp").reset_index(drop=True)

print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(f"\nDate range: {df_raw['Timestamp'].min()} — {df_raw['Timestamp'].max()}")
print(f"\nMissing values per column:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])


In [ ]:
df_raw.head(5)


In [ ]:
# Базовая статистика
quality_cols = [c for c in df_raw.columns if "[quality]" in c]
df_num = df_raw.drop(columns=quality_cols + ["Record number"]).set_index("Timestamp")
df_num.describe().round(3)


### 1.2 Очистка: дубликаты, регулярная сетка, интерполяция

In [ ]:
import sys
sys.path.insert(0, "src")
from pipeline import load_and_prepare

df = load_and_prepare(Path("data/brisbane_water_quality.csv"))
df.to_csv("data/prepared_temperature_ts.csv", index=False)
print(df[["ds", TARGET]].head(10))


### 1.3 EDA — Визуализация временного ряда

In [ ]:
y = df.set_index("ds")[TARGET]

fig, axes = plt.subplots(3, 1, figsize=(14, 9))

# Full series
axes[0].plot(y, lw=0.7, color="#1f77b4")
axes[0].set_title("Температура воды — полный ряд (2023-08-04 — 2023-10-06)")
axes[0].set_ylabel("°C")

# One week zoom
week_start = y.index[0]
week_end   = week_start + pd.Timedelta(days=7)
axes[1].plot(y[week_start:week_end], lw=1.2, color="#ff7f0e")
axes[1].set_title("Первые 7 дней (суточная сезонность)")
axes[1].set_ylabel("°C")

# Distribution
axes[2].hist(y.dropna(), bins=50, color="#2ca02c", edgecolor="white", alpha=0.8)
axes[2].set_title("Гистограмма Temperature")
axes[2].set_xlabel("°C")

plt.tight_layout()
plt.savefig("output/eda_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Среднее: {y.mean():.3f}°C | Std: {y.std():.3f} | Min: {y.min():.3f} | Max: {y.max():.3f}")


### 1.4 Декомпозиция временного ряда

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomp = seasonal_decompose(y.dropna(), model="additive", period=SEASON_LEN)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0]); axes[0].set_ylabel("Observed")
decomp.trend.plot(ax=axes[1], color="orange"); axes[1].set_ylabel("Trend")
decomp.seasonal.plot(ax=axes[2], color="green"); axes[2].set_ylabel("Seasonal")
decomp.resid.plot(ax=axes[3], color="red", alpha=0.7); axes[3].set_ylabel("Residual")
fig.suptitle(f"Seasonal Decomposition (period={SEASON_LEN})", fontsize=13)
plt.tight_layout()
plt.savefig("output/eda_decomposition.png", dpi=150, bbox_inches="tight")
plt.show()


### 1.5 Проверка стационарности

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

def test_stationarity(series, name):
    adf = adfuller(series.dropna(), autolag="AIC")
    kpss_res = kpss(series.dropna(), regression="c", nlags="auto")
    print(f"{'='*50}")
    print(f"Series: {name}")
    print(f"  ADF:  stat={adf[0]:.4f},  p={adf[1]:.4f}  =>",
          "STАЦИОНАРЕН" if adf[1] < 0.05 else "НЕ стационарен")
    print(f"  KPSS: stat={kpss_res[0]:.4f}, p={kpss_res[1]:.4f}  =>",
          "НЕ стационарен" if kpss_res[1] < 0.05 else "СТАЦИОНАРЕН")

test_stationarity(y, "Temperature (original)")
test_stationarity(y.diff().dropna(), "Temperature (1st diff)")
test_stationarity(y.diff(48).dropna(), "Temperature (seasonal diff=48)")


### 1.6 ACF / PACF

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6))
plot_acf(y.dropna(),  lags=96, ax=ax1, title="ACF — Temperature (96 lags = 2 days)")
plot_pacf(y.dropna(), lags=96, ax=ax2, title="PACF — Temperature")
plt.tight_layout()
plt.savefig("output/eda_acf_pacf.png", dpi=150, bbox_inches="tight")
plt.show()
print("""
ACF: медленное затухание => тренд присутствует.
Пики при lag=48, 96 => суточная сезонность (period=48).
PACF: резкий обрыв после 1-2 лагов => ARMA компонента p=1-2.
Рекомендация: использовать seasonal ARIMA(p,d,q)(P,D,Q)[48] или MSTL.
""")


## 2. Задача №3 (часть) — Обнаружение аномалий
Анализируем 3 метода: IQR, Z-score, Isolation Forest.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from scipy import stats

y_vals = y.dropna().values
y_idx  = y.dropna().index

# 1. IQR
Q1, Q3 = np.percentile(y_vals, 25), np.percentile(y_vals, 75)
IQR = Q3 - Q1
iqr_mask = (y_vals < Q1 - 1.5*IQR) | (y_vals > Q3 + 1.5*IQR)
iqr_anom = y_idx[iqr_mask]

# 2. Z-score
z_scores = np.abs(stats.zscore(y_vals))
z_mask   = z_scores > 3
z_anom   = y_idx[z_mask]

# 3. Isolation Forest
scaler  = StandardScaler()
y_s     = scaler.fit_transform(y_vals.reshape(-1, 1))
iso     = IsolationForest(contamination=0.02, random_state=SEED)
preds   = iso.fit_predict(y_s)
iso_mask = preds == -1
iso_anom = y_idx[iso_mask]

print(f"IQR аномалии:            {iqr_mask.sum()} точек")
print(f"Z-score (>3σ) аномалии:  {z_mask.sum()} точек")
print(f"Isolation Forest:        {iso_mask.sum()} точек")
print()
print("IQR — прост, не учитывает временную структуру.")
print("Z-score — предполагает нормальность, чувствителен к выбросам.")
print("Isolation Forest — ВЫБРАН как лучший: учитывает нелинейные зависимости,")
print("  не предполагает нормальность, хорошо работает с мультимодальными ВР.")


In [ ]:
# Визуализация аномалий
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(y, lw=0.7, color="#1f77b4", alpha=0.8, label="Temperature")
ax.scatter(iso_anom, y[iso_anom], color="red", s=25, zorder=5, label=f"Isolation Forest ({iso_mask.sum()})")
ax.scatter(iqr_anom, y[iqr_anom], color="orange", s=20, zorder=4, marker="^", label=f"IQR ({iqr_mask.sum()})")
ax.set_title("Обнаружение аномалий — сравнение методов")
ax.set_ylabel("Temperature °C")
ax.legend()
plt.tight_layout()
plt.savefig("output/anomaly_detection.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Задача №2 — Статистические методы (statsforecast)
5+ методов: Naive, SeasonalNaive, AutoARIMA, AutoETS, AutoTheta, MSTL  
Бектестинг: скользящее окно (n_windows=3, step=HORIZON)


In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import (
    AutoARIMA, AutoETS, AutoTheta, SeasonalNaive, Naive, MSTL
)

sf_df = df[["ds", TARGET]].copy()
sf_df.insert(0, "unique_id", "brisbane_temp")
sf_df = sf_df.rename(columns={TARGET: "y"})

models = [
    SeasonalNaive(season_length=SEASON_LEN),        # Baseline 1
    Naive(),                                          # Baseline 2
    AutoARIMA(season_length=SEASON_LEN),              # Авто ARIMA
    AutoETS(season_length=SEASON_LEN),                # Авто ETS
    AutoTheta(season_length=SEASON_LEN),              # Авто Theta
    MSTL(season_length=[SEASON_LEN, SEASON_LEN*7]),  # Multi-Seasonal
]

sf = StatsForecast(models=models, freq=FREQ, n_jobs=-1)
print("Запуск бектестинга (cross-validation)...")


In [ ]:
stat_cv = sf.cross_validation(
    df=sf_df, h=HORIZON, n_windows=N_WINDOWS, step_size=HORIZON
)
stat_cv.to_csv("output/stat_cv.csv", index=False)

# Метрики
from sklearn.metrics import mean_absolute_error, mean_squared_error

model_names = ["SeasonalNaive","Naive","AutoARIMA","AutoETS","AutoTheta","MSTL"]
rows = []
for m in model_names:
    if m not in stat_cv.columns: continue
    mask = ~stat_cv[m].isna()
    yt   = stat_cv.loc[mask, "y"].values
    yp   = stat_cv.loc[mask, m].values
    mae  = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mape = np.mean(np.abs((yt-yp)/(np.abs(yt)+1e-8)))*100
    rows.append({"Модель": m, "MAE": round(mae,4), "RMSE": round(rmse,4), "MAPE%": round(mape,4)})

stat_metrics = pd.DataFrame(rows).sort_values("MAE")
stat_metrics.to_csv("output/stat_metrics.csv", index=False)
print(stat_metrics.to_string(index=False))


In [ ]:
# Финальный прогноз
stat_fcst = sf.forecast(h=HORIZON, df=sf_df).reset_index()
stat_fcst.to_csv("output/stat_forecast.csv", index=False)

# Визуализация
best_stat = stat_metrics.iloc[0]["Модель"]
fig, ax = plt.subplots(figsize=(14, 4))
hist = sf_df.tail(96)
ax.plot(hist["ds"].values, hist["y"].values, label="Факт", lw=1.2)
for m in model_names:
    if m not in stat_fcst.columns: continue
    style = "-" if m == best_stat else "--"
    ax.plot(stat_fcst["ds"].values, stat_fcst[m].values,
            label=m, linestyle=style, lw=1.0 if m == best_stat else 0.8)
ax.set_title(f"Статистические модели (best: {best_stat})")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig("output/stat_forecast_plot.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Задача №3 — ML-модели (mlforecast)
LightGBM, Random Forest, XGBoost  
Feature engineering: лаги, скользящие средние, календарные признаки


In [ ]:
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

ml_df = sf_df.copy()  # unique_id, ds, y

ml_models = [
    LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=SEED, verbose=-1),
    RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
    xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=SEED, verbosity=0),
]

fcst = MLForecast(
    models=ml_models,
    freq=FREQ,
    lags=[1, 2, 48, 96],
    lag_transforms={
        1:  [("rolling_mean", 4), ("rolling_std", 4)],
        48: [("rolling_mean", 48)],
    },
    date_features=["hour", "dayofweek", "month"],
    target_transforms=[Differences([1])],
)

print("Fitting ML models...")
fcst.fit(ml_df)
print("Cross-validation...")
ml_cv = fcst.cross_validation(df=ml_df, h=HORIZON, n_windows=N_WINDOWS, step_size=HORIZON)
ml_cv.to_csv("output/ml_cv.csv", index=False)


In [ ]:
# ML метрики
ml_model_names = ["LGBMRegressor","RandomForestRegressor","XGBRegressor"]
ml_rows = []
for m in ml_model_names:
    if m not in ml_cv.columns: continue
    mask = ~ml_cv[m].isna()
    yt   = ml_cv.loc[mask, "y"].values
    yp   = ml_cv.loc[mask, m].values
    mae  = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mape = np.mean(np.abs((yt-yp)/(np.abs(yt)+1e-8)))*100
    ml_rows.append({"Модель": m, "MAE": round(mae,4), "RMSE": round(rmse,4), "MAPE%": round(mape,4)})

ml_metrics = pd.DataFrame(ml_rows).sort_values("MAE")
ml_metrics.to_csv("output/ml_metrics.csv", index=False)
print(ml_metrics.to_string(index=False))


In [ ]:
# ML прогноз + feature importance (LightGBM)
ml_fcst = fcst.predict(HORIZON)
ml_fcst.to_csv("output/ml_forecast.csv", index=False)

# Feature importance LightGBM
lgbm_model = fcst.models_["LGBMRegressor"]
fi = pd.Series(lgbm_model.feature_importances_,
               index=fcst.ts.features_order_).sort_values(ascending=False).head(15)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
best_ml = ml_metrics.iloc[0]["Модель"]
hist_ml = ml_df.tail(96)
ax1.plot(hist_ml["ds"].values, hist_ml["y"].values, label="Факт", lw=1.2)
for m in ml_model_names:
    if m not in ml_fcst.columns: continue
    ax1.plot(ml_fcst["ds"].values, ml_fcst[m].values,
             label=m, linestyle="--", lw=1.0)
ax1.set_title("ML прогнозы")
ax1.legend(fontsize=8)

fi.plot(kind="barh", ax=ax2, color="#1f77b4")
ax2.set_title("LightGBM Feature Importance (Top 15)")
ax2.invert_yaxis()
plt.tight_layout()
plt.savefig("output/ml_forecast_fi.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Задача №3 — DL-модели (neuralforecast)
NHITS, N-BEATS, PatchTST  
Все модели обучаются на CPU, max_steps=300


In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, NBEATS, PatchTST

INPUT_SIZE = HORIZON * 3

dl_train = ml_df.copy()  # unique_id, ds, y

dl_models = [
    NHITS(h=HORIZON, input_size=INPUT_SIZE, max_steps=300,
          accelerator="cpu", random_seed=SEED),
    NBEATS(h=HORIZON, input_size=INPUT_SIZE, max_steps=300,
           accelerator="cpu", random_seed=SEED),
    PatchTST(h=HORIZON, input_size=INPUT_SIZE, max_steps=300,
             accelerator="cpu", random_seed=SEED),
]

nf = NeuralForecast(models=dl_models, freq=FREQ)
print("Training DL models (может занять несколько минут)...")
nf.fit(dl_train)
dl_fcst = nf.predict()
dl_fcst.to_csv("output/dl_forecast.csv", index=False)
print("DL models done")
print(dl_fcst.head())


In [ ]:
# DL CV и метрики
print("Cross-validation DL models...")
dl_cv = nf.cross_validation(df=dl_train, n_windows=N_WINDOWS, step_size=HORIZON)
dl_cv.to_csv("output/dl_cv.csv", index=False)

dl_model_names = ["NHITS","NBEATS","PatchTST"]
dl_rows = []
for m in dl_model_names:
    if m not in dl_cv.columns: continue
    mask = ~dl_cv[m].isna()
    yt   = dl_cv.loc[mask, "y"].values
    yp   = dl_cv.loc[mask, m].values
    mae  = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mape = np.mean(np.abs((yt-yp)/(np.abs(yt)+1e-8)))*100
    dl_rows.append({"Модель": m, "MAE": round(mae,4), "RMSE": round(rmse,4), "MAPE%": round(mape,4)})

dl_metrics = pd.DataFrame(dl_rows).sort_values("MAE")
dl_metrics.to_csv("output/dl_metrics.csv", index=False)
print(dl_metrics.to_string(index=False))


## 6. Итоговая сравнительная таблица моделей

In [ ]:
# Объединяем все метрики
all_metrics = pd.concat([
    stat_metrics.assign(Тип="Статистическая"),
    ml_metrics.assign(Тип="ML"),
    dl_metrics.assign(Тип="DL"),
], ignore_index=True).sort_values("MAE")

all_metrics.index = range(1, len(all_metrics)+1)
print(all_metrics.to_string())
all_metrics.to_csv("output/all_metrics.csv", index=False)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {"Статистическая": "#1f77b4", "ML": "#ff7f0e", "DL": "#2ca02c"}
bar_colors = [colors[t] for t in all_metrics["Тип"]]

all_metrics.plot(x="Модель", y="MAE",  kind="bar", ax=axes[0],
                 color=bar_colors, legend=False, rot=45)
axes[0].set_title("MAE по моделям")

all_metrics.plot(x="Модель", y="RMSE", kind="bar", ax=axes[1],
                 color=bar_colors, legend=False, rot=45)
axes[1].set_title("RMSE по моделям")

from matplotlib.patches import Patch
legend_elems = [Patch(facecolor=v, label=k) for k, v in colors.items()]
axes[0].legend(handles=legend_elems, loc="upper right")
plt.tight_layout()
plt.savefig("output/models_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Задача №4 — Тестирование пайплайна

In [ ]:
import time
from sklearn.model_selection import TimeSeriesSplit

print("=" * 50)
print("Pipeline Performance Test")
print("=" * 50)

# Timing test
timings = {}

# Statistical (best model)
best_stat_model_cls = AutoETS if "AutoETS" in best_stat else AutoARIMA
t0 = time.time()
sf_test = StatsForecast(
    models=[AutoETS(season_length=SEASON_LEN)], freq=FREQ, n_jobs=1
)
sf_test.forecast(h=HORIZON, df=sf_df)
timings["AutoETS"] = round(time.time() - t0, 2)

# Best ML
t0 = time.time()
fcst.predict(HORIZON)
timings["LightGBM predict"] = round(time.time() - t0, 3)

print("\nВремя инференса (предсказание на H=48 шагов):")
for k, v in timings.items():
    print(f"  {k:25s}: {v:.3f}s")

# Residuals analysis (best stat model)
best_m = best_stat
resid = stat_cv["y"] - stat_cv[best_m] if best_m in stat_cv.columns else None

if resid is not None:
    from scipy import stats as scipy_stats
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    resid_clean = resid.dropna()
    axes[0].plot(resid_clean.values, lw=0.7, alpha=0.8)
    axes[0].axhline(0, color="red", linestyle="--")
    axes[0].set_title("Остатки")
    axes[1].hist(resid_clean, bins=40, color="#1f77b4", edgecolor="white")
    axes[1].set_title("Распределение остатков")
    scipy_stats.probplot(resid_clean, dist="norm", plot=axes[2])
    axes[2].set_title("QQ Plot (нормальность)")
    plt.suptitle(f"Анализ остатков — {best_m}", fontsize=12)
    plt.tight_layout()
    plt.savefig("output/residuals_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()

    lb_test = sm_diagnostic_acorr_ljungbox(resid_clean, lags=[10, 20], return_df=True) if False else None
    sw = scipy_stats.shapiro(resid_clean[:500])
    print(f"\nShapiro-Wilk (нормальность остатков): stat={sw[0]:.4f}, p={sw[1]:.4f}")
    print("  => Остатки", "нормальны" if sw[1] > 0.05 else "НЕ нормальны (p<0.05)")


## 8. Финальный вывод и сохранение результатов

In [ ]:
print("=" * 60)
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
print("=" * 60)
print(f"Датасет:     Brisbane Water Quality")
print(f"Таргет:      {TARGET}")
print(f"Горизонт:    {HORIZON} шагов ({HORIZON//2} часов)")
print(f"Метод CV:    скользящее окно ({N_WINDOWS} окна)")
print()
print("Лучшая статистическая модель:")
print(f"  {stat_metrics.iloc[0]['Модель']:20s} MAE={stat_metrics.iloc[0]['MAE']:.4f}")
print("Лучшая ML-модель:")
print(f"  {ml_metrics.iloc[0]['Модель']:20s} MAE={ml_metrics.iloc[0]['MAE']:.4f}")
print("Лучшая DL-модель:")
print(f"  {dl_metrics.iloc[0]['Модель']:20s} MAE={dl_metrics.iloc[0]['MAE']:.4f}")
print()
print("Все результаты сохранены в output/")
print("  - stat_metrics.csv, ml_metrics.csv, dl_metrics.csv")
print("  - all_metrics.csv")
print("  - *_forecast.csv, *_cv.csv")
print("  - anomalies_isolation_forest.csv")
print("  - графики: eda_*, anomaly_*, stat_*, ml_*, models_comparison.png")
